## This notebook checks location changes of (Ukrainian) Users based on the locations provided in the 2013, 2016, and 2019 data dumps.

### Key Take-Aways:

Ukraine:
- 14805 users are at least in one year located in Ukraine and joined gh before 2014
- Of the users providing their location in all three years:
    - 70% provide the exact same location in each year
    - 75% move within Ukraine
- Of the users providing their location in 2013, 2014, and 2016 and are located in Ukraine in at least one of these years:
    - 81% provide the exact same location in each year
    - 88% move within Ukraine
    
Russia:
- 44041 users are at least in one year located in Russia and joined gh before 2014
- Of the users providing their location in all three years:
    - 76% provide the exact same location in each year
    - 69% move within Russia
- Of the users providing their location in 2013, 2014 and 2016 and are located in Russia in at least one of these years:
    - 85% provide the exact same location in each year
    - 91% move within Russia

In [ ]:
# Import packages
!pip install google-cloud-bigquery
!pip install --upgrade google-cloud-bigquery[pandas]
import os, sys
from google.cloud import bigquery
import google.cloud.bigquery
import pandas as pd

In [ ]:
# Set path
ROOT = Path(__file__).resolve().parents[2]

DATA = ROOT / "database"
EXTRACTIONS = DATA / "extractions"
STATA = DATA / "stata"
OUTPUT = ROOT / "output"
FIGURES = OUTPUT / "figures"
TABLES = OUTPUT / "tables"

# Credentials 
credentials_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")

if not credentials_path:
    raise EnvironmentError(
        "GOOGLE_APPLICATION_CREDENTIALS is not set. "
        "Please set it to the path of your Google Cloud credentials JSON file."
    )

credentials_path = Path(credentials_path).expanduser().resolve()

if not credentials_path.exists():
    raise FileNotFoundError(
        f"Google Cloud credentials file not found: {credentials_path}"
    )

In [ ]:
def check_costs(query):
    # settings
    dry = bigquery.QueryJobConfig(dry_run = True, use_query_cache=False) 
    run = bigquery.QueryJobConfig(dry_run=False, use_query_cache=True)

    # check processing costs
    job = client.query(query, job_config=dry)
    print("Total GB that will be processed: ", job.total_bytes_processed/1000000000)
    print("Bytes billed: ", job.total_bytes_billed)

In [ ]:
# settings
dry = bigquery.QueryJobConfig(dry_run = True, use_query_cache=False) 
run = bigquery.QueryJobConfig(dry_run=False, use_query_cache=True)

### Locations of Ukrainian Users

In [ ]:
# Create a dataset containing user locations in different datadumps and select users who are in at least one year 
# located in Ukraine

query = """
# selecting users from Ukraine 
SELECT * FROM 
    (SELECT * FROM `ghtorrentmysql1906.MySQL1906.users`) u2019 #locations as of 2019, only country_codes
  LEFT JOIN 
    (SELECT login, name AS name_2013, email AS email_2013, company AS company_2013, 
    location AS location_2013
    FROM `friends-foes.user_panel.users2013` ) u2013 #locations as of 2013, locations need to be geocoded
  ON u2019.login = u2013.login
  LEFT JOIN 
    (SELECT login, name AS name_2013, email AS email_2014, company AS company_2014, 
    location AS location_2014
    FROM `friends-foes.user_panel.users2014`) u2014 #locations as of 2014, locations need to be geocoded
  ON u2019.login = u2014.login
  LEFT JOIN 
    (SELECT login, name, email, company AS company_2016, 
    location AS location_2016, country_code AS country_code2016, 
    city AS city_2016, long AS long_2016, lat AS lat_2016, form_address 
    FROM `friends-foes.user_panel.users2016` ) u2016 #locations as of 2016, country codes and unclassified self-reported
  ON u2019.login = u2016.login
  LEFT JOIN 
    (SELECT DISTINCT login,
        FIRST_VALUE(country_code)
          OVER (PARTITION BY id ORDER BY table_year) AS first_country, 
        FIRST_VALUE(long)
          OVER (PARTITION BY id ORDER BY table_year) AS first_long, 
        FIRST_VALUE(lat)
          OVER (PARTITION BY id ORDER BY table_year) AS first_lat,
        FIRST_VALUE(table_year)
          OVER (PARTITION BY id ORDER BY table_year) AS table_year,
        COUNT(DISTINCT country_code)
          OVER (PARTITION BY id) AS n_countries
        FROM `friends-foes.user_panel.ghuserpanel` ) ufl #first reported location from Thomas' panel
    ON u2019.login = ufl.login
    WHERE fake = 0 AND (lower(location) LIKE 'ua' OR lower(location) LIKE '%укр%' OR lower(location) LIKE '%ukra%' OR lower(location) LIKE '%ucra%' OR country_code LIKE 'ua'
    OR ufl.first_country = 'ua' 
    OR lower(u2016.location_2016) LIKE 'ua' OR lower(u2016.location_2016) LIKE '%укр%' OR lower(u2016.location_2016) LIKE '%ukra%' OR lower(u2016.location_2016) LIKE '%ucra%' OR u2016.country_code2016 LIKE 'ua' 
    OR lower(u2013.location_2013) LIKE 'ua' OR lower(u2013.location_2013) LIKE '%укр%' OR lower(u2013.location_2013) LIKE '%ukra%' OR lower(u2013.location_2013) LIKE '%ucra%'
    OR lower(u2014.location_2014) LIKE 'ua' OR lower(u2014.location_2014) LIKE '%укр%' OR lower(u2014.location_2014) LIKE '%ukra%' OR lower(u2014.location_2014) LIKE '%ucra%')
ORDER BY created_at
"""
check_costs(query) # saved on big query as ua_user_panel

In [ ]:
query = """
SELECT *
FROM `friends-foes.user_panel.ua_user_panel`
ORDER BY created_at
"""
check_costs(query)

In [ ]:
job = client.query(query, job_config=run) 
df = job.to_dataframe()
print(len(df))

In [ ]:
# select users who joined gh before 2014
df['year_joingh'] = pd.DatetimeIndex(df['created_at']).year
before = df[df['year_joingh'] < 2014]
print(len(before))

In [ ]:
#df.info()

#### Location measures in the dataset

- 2013: location_2013
- 2014: location_2014
- 2016: location_2016, country_code2016
- 2019: location, country_code, state, city

In [ ]:
df[['location_2013','location_2014','location_2016','location','state','city']].head(50)

In [ ]:
# are location_2013 and location_2014 exactly the same? Answer: No
temp = df[df['location_2013'] != df['location_2014']]
len(temp)

In [ ]:
# dataset with users proving locations in all years
temp = df[(df['location_2013'].notnull()) & (df['location_2014'].notnull()) & (df['location_2016'].notnull()) & (df['location'].notnull())]
len(temp)

In [ ]:
# users with exact same location in all years
x = len(temp[(temp['location_2013'] == temp['location_2016']) & (temp['location_2013'] == temp['location_2014']) & (temp['location_2013'] == temp['location'])])
print('Users with exact same location in Ukraine in 13, 14, 16, 19: %0.0f' % x)
y = x/len(temp)
print('')
print('Exact same location/ all users providing all locations: % 0.2f' % y)

In [ ]:
# users moving only within Ukraine, providing locations always

# user needs to have "UA" or "Ukraine" or "ua" or "ukraine" in all three locations
ua_identifier = 'Ukrai|ukrai|ua|UA|UA|Kiev|Kyiv'

x = len(before[(before['location_2013'].str.contains(ua_identifier)) & 
              (before['location_2014'].str.contains(ua_identifier)) & 
              (before['location_2016'].str.contains(ua_identifier)) &
              (before['location'].str.contains(ua_identifier))])
print('Users in Ukraine in 13, 14, 16, 19: %0.0f' % x)
x = x/len(temp)
print('')
print('In Ukraine/ at least once Ukraine (given all locations provided): % 0.2f' % x)
x = 1 - x
print('')
print('Share of users moving outside Ukraine: % 0.2f' % x)

In [ ]:
# dataset with users in UA in at least one year of 13, 14 and 16 proving locations in 2013, 2014, AND 2016
temp = df[(df['location_2013'].notnull()) & (df['location_2014'].notnull()) & (df['location_2016'].notnull())]
temp = temp[(temp['location_2013'].str.contains(ua_identifier)) |
        (temp['location_2014'].str.contains(ua_identifier)) |
        (temp['location_2016'].str.contains(ua_identifier))]
len(temp)

In [ ]:
# users with exact same location in 2013, 2014, and 2016
x = len(temp[(temp['location_2013'] == temp['location_2014']) & (temp['location_2013'] == temp['location_2016'])])
print('Users with exact same location in Ukraine in 13, 14, 16: %0.0f' % x)
x = x/len(temp)
print('')
print('Exact same location/ all users providingy locations in 13, 14, and 16: % 0.2f' % x)

In [ ]:
# users moving only within Ukraine, providing locations in 2013 and 2016

# user needs to have "UA" or "Ukraine" or "ua" or "ukraine" in all three locations
ua_identifier = 'Ukrai|ukrai|ua|UA|UA|Kiev|Kyiv'

x = len(temp[(temp['location_2013'].str.contains(ua_identifier)) & 
              (temp['location_2014'].str.contains(ua_identifier)) & 
              (temp['location_2016'].str.contains(ua_identifier))])
print('Users in Ukraine in 13, 16: %0.0f' % x)
x = x/len(temp)
print('')
print('In Ukraine/ at least once Ukraine (given all three locations provided): % 0.2f' % x)
x = 1 - x
print('')
print('Share of users moving outside Ukraine: % 0.2f' % x)

### Locations of Russian Users

In [ ]:
# Create a dataset containing user locations in different datadumps and select users who are in at least one year 
# located in Ukraine

query = """
# selecting users from Russia
# save result here in cloud as: ru_user_panel

SELECT * FROM 
    (SELECT * FROM `ghtorrentmysql1906.MySQL1906.users`) u2019 #locations as of 2019, only country_codes
  LEFT JOIN 
    (SELECT login, name, email, company AS company_2013, 
    location AS location_2013
    FROM `friends-foes.user_panel.users2013` ) u2013 #locations as of 2013, country codes and unclassified self-reported
  ON u2019.login = u2013.login
  LEFT JOIN 
    (SELECT login, name AS name_2013, email AS email_2014, company AS company_2014, 
    location AS location_2014
    FROM `friends-foes.user_panel.users2014`) u2014 #locations as of 2014, locations need to be geocoded
  ON u2019.login = u2014.login
  LEFT JOIN 
    (SELECT login, name, email, company AS company_2016, 
    location AS location_2016, country_code AS country_code2016, 
    city AS city_2016, long AS long_2016, lat AS lat_2016, form_address 
    FROM `friends-foes.user_panel.users2016` ) u2016 #locations as of 2016, country codes and unclassified self-reported
  ON u2019.login = u2016.login
  LEFT JOIN 
    (SELECT DISTINCT login,
        FIRST_VALUE(country_code)
          OVER (PARTITION BY id ORDER BY table_year) AS first_country, 
        FIRST_VALUE(long)
          OVER (PARTITION BY id ORDER BY table_year) AS first_long, 
        FIRST_VALUE(lat)
          OVER (PARTITION BY id ORDER BY table_year) AS first_lat,
        FIRST_VALUE(table_year)
          OVER (PARTITION BY id ORDER BY table_year) AS table_year,
        COUNT(DISTINCT country_code)
          OVER (PARTITION BY id) AS n_countries
        FROM `friends-foes.user_panel.ghuserpanel` ) ufl #first reported location from Thomas' panel
    ON u2019.login = ufl.login
    WHERE fake = 0 AND (lower(location) LIKE 'russia' OR country_code LIKE 'ru' OR country_code LIKE 'rus'
    OR ufl.first_country = 'ru' 
    OR lower(u2016.location_2016) LIKE 'ru' OR lower(u2016.location_2016) LIKE '%russi%' OR lower(u2016.location_2016) LIKE '%rossi%'
      OR lower(u2016.location_2016) LIKE '%рф%' OR lower(u2016.location_2016) LIKE '%росс%' OR u2016.country_code2016 LIKE 'ru' 
    OR lower(u2013.location_2013) LIKE 'ru' OR lower(u2013.location_2013) LIKE '%russi%' OR lower(u2013.location_2013) LIKE '%rossi%'
      OR lower(u2013.location_2013) LIKE '%рф%' OR lower(u2013.location_2013) LIKE '%росс%' 
    OR lower(u2014.location_2014) LIKE 'ru' OR lower(u2014.location_2014) LIKE '%russi%' OR lower(u2014.location_2014) LIKE '%rossi%'
      OR lower(u2014.location_2014) LIKE '%рф%' OR lower(u2014.location_2014) LIKE '%росс%' 
      )
ORDER BY created_at
"""
check_costs(query) # saved on big query as ua_user_panel

In [ ]:
query = """
SELECT *
FROM `friends-foes.user_panel.ru_user_panel`
ORDER BY created_at
"""
check_costs(query)

In [ ]:
job = client.query(query, job_config=run) 
df = job.to_dataframe()
print(len(df))

In [ ]:
df[['location_2013','location_2014','location_2016','location','state','city']].head(50)

In [ ]:
# delete Prussia
df = df[df['location_2016'].str.contains('Prussia') == False]

In [ ]:
# select users who joined gh before 2014
df['year_joingh'] = pd.DatetimeIndex(df['created_at']).year
before = df[df['year_joingh'] < 2014]
print(len(before))

In [ ]:
# dataset with users proving locations in all years
temp = df[(df['location_2013'].notnull()) & (df['location_2014'].notnull()) & (df['location_2016'].notnull()) & (df['location'].notnull())]
len(temp)

In [ ]:
# users with exact same location in all years
x = len(temp[(temp['location_2013'] == temp['location_2016']) & (temp['location_2013'] == temp['location_2014']) & (temp['location_2013'] == temp['location'])])
print('Users with exact same location in Russia in 13, 14, 16, 19: %0.0f' % x)
y = x/len(temp)
print('')
print('Exact same location/ all users providing all locations: % 0.2f' % y)

In [ ]:
# users moving only within Russia, providing locations always

# user needs to have "RU" or "Russia" or "ru" or "russia" in all three locations
ru_identifier = 'Ru|ru|Moscow'

x = len(before[(before['location_2013'].str.contains(ru_identifier)) & 
              (before['location_2014'].str.contains(ru_identifier)) & 
              (before['location_2016'].str.contains(ru_identifier)) &
              (before['location'].str.contains(ru_identifier))])
print('Users in Russia in 13, 14, 16, 19: %0.0f' % x)
x = x/len(temp)
print('')
print('In Ukraine/ at least once Russia (given all locations provided): % 0.2f' % x)
x = 1 - x
print('')
print('Share of users moving outside Russia: % 0.2f' % x)

In [ ]:
# dataset with users in RU in at least one year of 13,14, and 16 proving (non-None) locations in 2013, 2014, and 2016
temp = df[(df['location_2013'].notnull()) & (df['location_2014'].notnull()) & (df['location_2016'].notnull())]
temp = temp[(temp['location_2013'].str.contains(ru_identifier)) |
        (temp['location_2014'].str.contains(ru_identifier)) |
        (temp['location_2016'].str.contains(ru_identifier))]
len(temp)

In [ ]:
# users with exact same location in 2013, 2014, and 2016
x = len(temp[(temp['location_2013'] == temp['location_2014']) & (temp['location_2013'] == temp['location_2016'])])
print('Users with exact same location in Russia in 13, 14, 16: %0.0f' % x)
x = x/len(temp)
print('')
print('Exact same location/ all users providingy locations in 13, 14, and 16: % 0.2f' % x)

In [ ]:
# users moving only within Ukraine, providing locations in 2013 and 2016

# user needs to have "UA" or "Ukraine" or "ua" or "ukraine" in all three locations
ru_identifier = 'Ru|ru|Moscow'

x = len(temp[(temp['location_2013'].str.contains(ru_identifier)) & 
              (temp['location_2014'].str.contains(ru_identifier)) & 
              (temp['location_2016'].str.contains(ru_identifier))])
print('Users in Ukraine in 13, 16: %0.0f' % x)
x = x/len(temp)
print('')
print('In Ukraine/ at least once Ukraine (given all three locations provided): % 0.2f' % x)
x = 1 - x
print('')
print('Share of users moving outside Ukraine: % 0.2f' % x)